# 01 — rung 19b: the epoch sweep on the FULL 6,252

The external-data arm's **five** checkpoints, answered on the whole `frame_ood_v1`
validation set — **38 videos / 6,252 questions**. No training, no new data, inference only.
Chained off the end of training by `_tools/chain_eval.py`.

**Two reads, and they are not the same question.**

1. **The rung's question — the corpus.** `19b_ep4 − 47_ep4`, paired and video-clustered.
   Both arms trained on zero of these 38 videos, both anneal a cosine over five epochs,
   both were merged and scored on this box through this code path. `--dataset` is the
   only difference (`diff_vs_r47.json`, gate 2).
2. **The curve — which epoch is best.** Within-arm `ep(n+1) − ep(n)` for all five. Rung 42's
   entire advantage turned out to be **epochs, not corpus**, and rung 47's own curve goes
   flat after ep4 (+0.0030, CI spans zero). If 5,718 external rows change the *shape* of
   that curve and not just its level, that is a different finding from a level shift.

🔴 **ep1 and ep2 have no rung-47 control on the 6,252** — only ep3/ep4/ep5 were re-scored
there. They are within-arm trajectory and are never paired across arms.

🔴 **The 6,252 is not the centre axis.** It is the same 38 videos for both arms, but this
arm has now met Strasbourg. The paired delta stays valid; a raw 19b number does not mean
what rung 47's means. Rung 48's `bag_f1` over the 15 `hold` videos is the centre axis, is
reported separately, and is **never averaged with `bucket_mean`** — that averaging is the
dilution rung 48 v2 caught.


In [ ]:
# --- bootstrap -------------------------------------------------------------------
# 🔴 NOTHING that touches HuggingFace or `frame` may be imported here. `HF_HOME` has to be
# set before the first HF import, and its value comes from the PARAMETERS cell below.
import json, logging, os, re, sys, time
from pathlib import Path
import pandas as pd

# 🔴 a papermill kernel does NOT inherit the env's bin/ on PATH (rung 39's scar)
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")
print("python:", sys.executable)


In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) -------------
SMOKE   = True
EPOCHS  = [1, 2, 3, 4, 5]
N_BOOT  = 4000
KEEP_MERGED = False
STORAGE = "/mnt/storage/uaq_user"


In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) -----
os.environ["HF_HOME"] = f"{STORAGE}/hf_cache"
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

REPO = f"{STORAGE}/repo_leo"
for _t in ("45-gen36-data-and-reg", "47-epochs-vs-corpus", "19b-external-recognition"):
    sys.path.insert(0, f"{REPO}/experiments/{_t}/_tools")
from eval_arm45 import ensure_paths
ensure_paths(REPO)
import eval_arm45 as E45
import eval_arm19b as E
import transformers

cfg = E.Rung19bConfig(
    repo_root=REPO, work_root=f"{STORAGE}/rung19b", data_root=f"{STORAGE}/orena-data",
    frames_cache=f"{STORAGE}/frames_cache", hf_home=f"{STORAGE}/hf_cache",
    corpus=f"{STORAGE}/rung48/corpus/train_merged.jsonl",
    epochs=tuple(EPOCHS), n_boot=N_BOOT, keep_merged=KEEP_MERGED, smoke=SMOKE,
)
EXP = Path(REPO) / "experiments" / "19b-external-recognition"
MANIFEST = Path(REPO) / "experiments" / "splits" / "frame_ood_v1.csv"
CHOLECT50_SPLIT = Path(REPO) / "experiments" / "splits" / "cholect50_split_v1.csv"

CKPT_ROOT = E.resolve_ckpt_root(cfg)
print("swift    :", (Path(_envbin) / "swift").exists())
print("tfmrs    :", transformers.__version__)
print("arm      :", E.RUN, "· epochs", EPOCHS, "-> steps", [E.EPOCH_STEPS[e] for e in EPOCHS])
print("ckpt_root:", CKPT_ROOT)
print("out_dir  :", cfg.eval_root)


In [ ]:
# --- GATE 1: the run actually moved weights, and reached the last epoch asked for ---
# 🔴 rc=0 is NOT evidence. AdamW's decoupled weight decay moves every tensor at zero
# gradient, so a checkpoint diff cannot separate a real run from a no-op — only the
# grad_norm log can ([[rc-zero-is-not-evidence]]).
train_gate = E.assert_run_moved_weights(CKPT_ROOT)
print(json.dumps(train_gate, indent=1, default=str))

# This notebook is CHAINED off the end of training, so it must not assume the run got
# there. Every epoch asked for has to be on disk, complete, before a GPU-second is spent.
for e in EPOCHS:
    E.assert_checkpoint_complete(CKPT_ROOT, e)
if max(EPOCHS) == 5 and not train_gate["run_finished"]:
    raise AssertionError(
        "ep5 was requested but the trainer never wrote its end-of-run trailer. The last "
        f"checkpoint may be mid-flush. last_epoch_logged={train_gate['last_epoch_logged']}")
print(f"OK train gate: {len(EPOCHS)} checkpoints complete · finished={train_gate['run_finished']}")


In [ ]:
# --- GATE 2: `--dataset` is still the only variable vs rung 47. RAISES. ------------
diff = E.assert_single_variable(cfg.run_dir)
print("single variable:", json.dumps(diff, indent=1))


In [ ]:
# --- GATE 3: the corpus touched NONE of the 38, and NONE of the 15 `hold`. RAISES. --
# The premise the whole notebook rests on, measured here rather than quoted from the
# launcher's `gates.json`. A gate that only ever runs once is a claim.
norm = lambda s: re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")

man = pd.read_csv(MANIFEST, dtype={"dataset": str, "video_id": str})
man["key"] = man.dataset.map(norm) + "__" + man.video_id.map(norm)
by = {s: set(g.key) for s, g in man.groupby("split")}
EVAL_KEYS = by["val_id"] | by["val_ood"]

legality = E.assert_corpus_legality(cfg.corpus, EVAL_KEYS, by["train"], str(CHOLECT50_SPLIT))
print(json.dumps(legality, indent=1))
print(f"OK legality: {legality['challenge_videos']} challenge + "
      f"{legality['external_videos']} CholecT50 videos · "
      f"0 of {len(EVAL_KEYS)} eval videos · 0 of the 15 `hold`")


In [ ]:
# --- GATE 4: the eval set, and that every frame it needs is cached. RAISES. --------
from frame.config import BaselineConfig
from frame.data import load_frame_items

items = load_frame_items(BaselineConfig(data_root=Path(cfg.data_root)))
eval_items = [i for i in items if norm(i.dataset) + "__" + norm(i.video_id) in EVAL_KEYS]
BRIDGE_QIDS = {i.request.qID for i in eval_items}   # qID lives on the SDK Request
BRIDGE_VIDEOS = {(i.dataset, i.video_id) for i in eval_items}

if len(BRIDGE_QIDS) != 6252:
    raise AssertionError(f"expected 6,252 questions, got {len(BRIDGE_QIDS)}")
if len(BRIDGE_VIDEOS) != 38:
    raise AssertionError(f"expected 38 videos, got {len(BRIDGE_VIDEOS)}")

split = {"held_videos": BRIDGE_VIDEOS, "held_qids": BRIDGE_QIDS,
         "n_held": len(BRIDGE_QIDS), "n_heico_videos": len(by["val_ood"])}

cov = E45.assert_cache_covers(cfg.to_eval_config(EPOCHS[0]), eval_items)
print(f"OK eval set: {len(BRIDGE_VIDEOS)} videos / {len(BRIDGE_QIDS)} questions · cache {cov}")


In [ ]:
# --- GATE 5: the judge resolves offline, and rung 47's controls are on disk. -------
# Both fail LATE and expensively if left unchecked: the judge ~30 min into an eval with
# the model already loaded, the controls only when the paired CI is formed, after every
# GPU-second has been spent.
from transformers import AutoTokenizer
_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}.") from exc
print(f"OK judge gate: {_judge}")

CTRL = {}
for e in sorted(set(EPOCHS) & set(E.CONTROL_47_EPOCHS)):
    p = E.control_47_csv(e)
    n = sum(1 for _ in open(p, encoding="utf-8")) - 1
    if n != len(BRIDGE_QIDS):
        raise AssertionError(f"rung-47 ep{e} control has {n} rows, not {len(BRIDGE_QIDS)}")
    CTRL[e] = p
print(f"OK controls: rung 47 ep{sorted(CTRL)} · {len(BRIDGE_QIDS)} rows each")
print(f"   no control for ep{sorted(set(EPOCHS) - set(CTRL))} — within-arm trajectory only")


In [ ]:
# --- the SWEEP: merge -> answer the 6,252 -> reclaim the 16 GB, five times ---------
from frame import ledger

gold = ledger.gold_from_frame_parquets(Path(cfg.data_root))
ARM, TIMING = {}, {}
for e in EPOCHS:
    t0 = time.perf_counter()
    print(f"=== epoch {e} (checkpoint-{E.EPOCH_STEPS[e]}) ===", flush=True)
    csv_path = E.answer_epoch(cfg, CKPT_ROOT, e, split, len(BRIDGE_QIDS))
    scored = E.score_on_heldout(cfg, csv_path, BRIDGE_QIDS, gold, f"ep{e}")
    ARM[e] = {"epoch": e, "step": E.EPOCH_STEPS[e], "cells": E.cells(scored["strat"]),
              "res": scored["res"], "csv": str(csv_path)}
    TIMING[e] = time.perf_counter() - t0
    print(f"    ep{e} bucket_mean={ARM[e]['cells']['bucket_mean']:.4f} "
          f"({TIMING[e]/60:.1f} min)", flush=True)


In [ ]:
# --- 🎯 read 2: the epoch curve — is any epoch better than another? ----------------
from frame import metrics as M

def paired(res_a, res_b, label, *, strict=True):
    """Video-clustered paired CI of (a − b) in the 6-cell shape. RULES §13.

    Clusters on VIDEO, not on question — the correction that closed rung 47's CI.
    """
    for nm, r in (("a", res_a), ("b", res_b)):
        for col in ("qID", "video", "correctness"):
            if col not in r.columns:
                raise AssertionError(f"{nm} is missing {col!r}; columns={list(r.columns)}")
    m = res_a.merge(res_b, on="qID", suffixes=("_a", "_b"))
    if strict and not SMOKE and len(m) != len(BRIDGE_QIDS):
        raise AssertionError(f"paired {len(m)} of {len(BRIDGE_QIDS)} — the arms answered "
                             "different question sets")
    if len(m) != min(len(res_a), len(res_b)):
        raise AssertionError(f"paired {len(m)} but arms have {len(res_a)}/{len(res_b)} rows")
    m["video"] = m["video_a"]
    g = (m["primary_a"] if "primary_a" in m else m["primary_capability_a"]).map(M._leaf_to_group)
    d = m["qID"].map(M._dist_from_qid)
    out = []
    for dd in ("ID", "OOD"):
        for gg in sorted(set(g.dropna())) + ["ALL"]:
            sl = m[(d == dd) & ((g == gg) if gg != "ALL" else True)]
            r = M.paired_delta_ci(sl, correct_a="correctness_a", correct_b="correctness_b",
                                  n_boot=N_BOOT, seed=cfg.seed)
            out.append({"comparison": label, "cell": f"{gg}_{dd}", "n": r["n"],
                        "videos": r["n_videos"], "delta": round(r["delta"], 4),
                        "ci_low": round(r["ci_low"], 4), "ci_high": round(r["ci_high"], 4),
                        "excludes_0": bool(r["ci_low"] > 0 or r["ci_high"] < 0)})
    return out

curve_ci = []
for a, b in zip(sorted(ARM)[1:], sorted(ARM)[:-1]):
    curve_ci += paired(ARM[a]["res"], ARM[b]["res"], f"19b ep{a} - ep{b}")
curve_df = pd.DataFrame(curve_ci)
HEADLINE = ["ALL_ID", "ALL_OOD", "object_recognition_ID", "object_recognition_OOD"]
# A one-epoch sweep (a smoke) has no adjacent pair. That is not a failure, and it must
# not crash the cell that still has the cross-arm read to do.
if curve_df.empty:
    print("no adjacent epoch pair to compare — the curve needs at least two epochs")
else:
    print(curve_df[curve_df.cell.isin(HEADLINE)].to_string(index=False))


In [ ]:
# --- 🎯 read 1: the rung's question — the CORPUS, at matched epochs ----------------
# 🔴 Only ep3/ep4/ep5 exist for rung 47 on this eval set. ep4 is the pre-registered one
# (README); ep3 and ep5 are reported beside it so a single-epoch coincidence is visible.
corpus_ci = []
for e in sorted(CTRL):
    ctrl = pd.read_csv(CTRL[e])
    ctrl = ctrl[ctrl["qID"].isin(BRIDGE_QIDS)].copy()
    corpus_ci += paired(ARM[e]["res"], ctrl, f"19b ep{e} - 47 ep{e}")
corpus_df = pd.DataFrame(corpus_ci)
if corpus_df.empty:
    raise AssertionError("no epoch with a rung-47 control was scored — this rung's own "
                         "question is unanswered. Score at least ep4.")
print(corpus_df[corpus_df.cell.isin(HEADLINE)].to_string(index=False))

d = E.DECISIVE_EPOCH
decisive = corpus_df[corpus_df.comparison == f"19b ep{d} - 47 ep{d}"]
print("\n--- the pre-registered cell ---")
if decisive.empty:
    print(f"⚠️ ep{d} was NOT scored. Everything above is context; the rung's question "
          "is still open.")
else:
    print(decisive[decisive.cell.isin(["ALL_ID", "ALL_OOD"])].to_string(index=False))


In [ ]:
# --- write, and say plainly what these numbers are NOT -----------------------------
rows = [{"arm": "19b_external (A2 + 5,718 Strasbourg)", "eval_set": "full 6252 (38 videos)",
         "epoch": e, "ckpt": f"checkpoint-{ARM[e]['step']}",
         "minutes": round(TIMING[e] / 60, 1),
         **{k: round(v, 4) for k, v in ARM[e]["cells"].items()}}
        for e in sorted(ARM)]
tbl = pd.DataFrame(rows)
print(tbl.drop(columns=["arm", "eval_set"]).to_string(index=False))

best = max(ARM, key=lambda e: ARM[e]["cells"]["bucket_mean"])
print(f"\nbest bucket_mean: ep{best} = {ARM[best]['cells']['bucket_mean']:.4f}")
print("⚠️ 'best' here is a POINT estimate. Whether it beats its neighbour is the CI table "
      "above, not this line — rung 47's ep5 led ep4 by +0.0030 and the interval spanned 0.")

ctrl_rows = [{"arm": "47_C_epochs (A2 corpus)", "eval_set": "full 6252 (38 videos)",
              "epoch": e, "ckpt": "-", "minutes": None, **E.CONTROL_47[e]}
             for e in sorted(E.CONTROL_47)]
print("\n--- rung 47, same eval set (control, transcribed) ---")
print(pd.DataFrame(ctrl_rows).drop(columns=["arm", "eval_set", "ckpt", "minutes"]).to_string(index=False))

if not SMOKE:
    tbl.to_csv(EXP / "RESULTS_epochs_full6252.csv", index=False)
    curve_df.to_csv(EXP / "RESULTS_epoch_curve_paired_ci.csv", index=False)
    corpus_df.to_csv(EXP / "RESULTS_corpus_vs_r47_paired_ci.csv", index=False)
    print("\nwritten:", EXP / "RESULTS_epochs_full6252.csv")
    print("        ", EXP / "RESULTS_epoch_curve_paired_ci.csv")
    print("        ", EXP / "RESULTS_corpus_vs_r47_paired_ci.csv")


In [ ]:
# --- what is still NOT measured here ----------------------------------------------
# 🔴 This notebook produces ZERO evidence about CENTRE. The 6,252 are the challenge's own
# videos; the 15 CholecT50 `hold` videos are rung 48's axis and are scored by rung 48's
# `bag_f1`, on its own ruler, and are NEVER averaged into `bucket_mean`.
#
# 🔴 And for THIS arm the 15 `hold` mean something different than they did for rung 47:
# rung 47 had never met Strasbourg, so for it they were UNSEEN CENTRE. This arm has met
# 35 of the 50, so for it they are UNSEEN SCENE within a seen centre. The paired delta
# survives that; the interpretation of either arm's own number does not. Say which one
# you are reporting (README, "How it is read").
print("sweep complete —", len(ARM), "epochs ·", f"{sum(TIMING.values())/3600:.2f} h")
